## RAG Day 3

### Expert Question Answerer for InsureLLM

LangChain 1.0 implementation of a RAG pipeline.

Using the VectorStore we created last time (with HuggingFace `all-MiniLM-L6-v2`)

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

ImportError: cannot import name 'ModelAPIError' from 'langchain_core.exceptions' (/Users/animishpande/MyPC/Work/LLM_Course/Forked/llm_engineering/.venv/lib/python3.12/site-packages/langchain_core/exceptions.py)

In [ ]:
# MODEL = "gpt-4.1-nano"
MODEL = "gemini-3.1-flash-lite"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

### Connect to Chroma; use Hugging Face all-MiniLM-L6-v2

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### Set up the 2 key LangChain objects: retriever and llm

#### A sidebar on "temperature":
- Controls how diverse the output is
- A temperature of 0 means that the output should be predictable
- Higher temperature for more variety in answers

Some people describe temperature as being like 'creativity' but that's not quite right
- It actually controls which tokens get selected during inference
- temperature=0 means: always select the token with highest probability
- temperature=1 usually means: a token with 10% probability should be picked 10% of the time

Note: a temperature of 0 doesn't mean outputs will always be reproducible. You also need to set a random seed. We will do that in weeks 6-8. (Even then, it's not always reproducible.)

Note 2: if you want creativity, use the System Prompt!

In [ ]:
retriever = vectorstore.as_retriever()
# llm = ChatOpenAI(temperature=0, model_name=MODEL)
llm = ChatGoogleGenerativeAI(temperature=1, model=MODEL)

### These LangChain objects implement the method `invoke()`

## What does invoke do?
- The retriever is a variable that is assigned vectorstore.as_retriever. Here it takes input as string, converts that string into a vector embedding using the Huggingface embeddings model that was provided to the vector store when initializing. After converting the string input into vector embeddings, it retrieves all the vectors that are closest to the input in the Chroma DB. This is how the invoke function works.

In [8]:
retriever.invoke("Who is Avery?")

[Document(id='164fc381-b9af-437a-bf64-b8488bc8aa53', metadata={'doc_type': 'employees', 'source': 'knowledge-base/employees/Avery Lancaster.md'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and ada

# What does LLM Invoke do?
- The LLM invoke is just an abstraction that retrieves data from the AI model which we have chosen.

In [14]:
llm.invoke("Who is Avery?")

AIMessage(content=[{'type': 'text', 'text': 'Because "Avery" is a common name, it could refer to several different people or entities depending on the context. Here are the most likely possibilities:\n\n### 1. Fictional Characters\n*   **Avery Bullock:** A recurring character in the animated series *American Dad!*. He is the eccentric and often unhinged Deputy Director of the CIA.\n*   **Avery Jessup:** A character from the TV show *30 Rock*, played by Elizabeth Banks. She is a high-powered news anchor and the wife of Jack Donaghy.\n*   **Avery Barkley:** A character from the musical drama series *Nashville*, played by Jonathan Jackson.\n*   **Avery Samuels:** A character from the horror anthology series *Scream Queens*.\n\n### 2. Historical Figures\n*   **Oswald Avery:** A famous Canadian-American physician and medical researcher. He is best known for the "Avery–MacLeod–McCarty experiment" (1944), which proved that DNA is the substance that causes genetic transformation, a foundationa

## Time to put this together!

In [15]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [24]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    # return response.content
    return response.content[0]["text"]


In [25]:
answer_question("Who is Averi Lancaster?", [])

"Hello! I'd be happy to tell you about Avery Lancaster.\n\nAvery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. Based in San Francisco, she has been leading the company since she co-founded it in 2015. Under her guidance, Insurellm has grown into a leading Insurance Tech provider. Avery is widely recognized for her innovative leadership strategies and her deep expertise in risk management, which have been instrumental in bringing the company into the mainstream insurance market.\n\nBefore founding Insurellm, Avery served as a Senior Product Manager at Innovate Insurance Solutions from 2013 to 2015, where she focused on developing insurance products for the tech sector.\n\nIs there anything else you'd like to know about our leadership team?"

## What could possibly come next? 😂

In [26]:
gr.ChatInterface(answer_question).launch()

/Users/animishpande/MyPC/Work/LLM_Course/Forked/llm_engineering/.venv/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## Admit it - you thought RAG would be more complicated than that!!